# ALMA Published Wind Rose Plots

This notebook generates polar scatter wind rose plots for the ALMA ground-based meteorological data (`alma_published/*.met` files).

**Plot style** matches `structures/plots.ipynb`:
- Each data point is one scatter dot on a polar axis
- **Radius** = wind speed (m/s)
- **Angle** = wind direction (meteorological: N=0°, clockwise)
- **Color** = wind speed (viridis colormap)

**Output:** 25 plots total
- 24 monthly plots: `wsb_0101` → Jan 2001 … `wsb_0212` → Dec 2002
- 1 combined plot: all data from both years merged

In [ ]:
# ============================================================
# IMPORTS & CONFIGURATION
# ============================================================
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# ── Path to the published .met files ──────────────────────────
ALMA_DIR = os.path.abspath(r"..\data\raw\alma_published")

# ── Column indices (0-based) from config.py ──────────────────
COL_TIMESTAMP = 0
COL_SPEED     = 7   # wind speed in m/s
COL_DIRECTION = 10  # wind direction in degrees (meteorological)
MISSING       = -999

# ── Plot style ────────────────────────────────────────────────
CMAP      = "viridis"
DOT_SIZE  = 8
ALPHA     = 0.7

# Month names for titles
MONTH_NAMES = {
    '01': 'January',  '02': 'February', '03': 'March',
    '04': 'April',    '05': 'May',       '06': 'June',
    '07': 'July',     '08': 'August',    '09': 'September',
    '10': 'October',  '11': 'November',  '12': 'December'
}

# Year mapping: filename prefix → calendar year
YEAR_MAP = {'01': '2001', '02': '2002'}

print(f"Looking for .met files in: {ALMA_DIR}")
files = sorted(glob.glob(os.path.join(ALMA_DIR, "wsb_*.met")))
print(f"Found {len(files)} files:")
for f in files:
    print(" ", os.path.basename(f))

In [ ]:
# ============================================================
# HELPER: parse a single .met file → (speed_array, direction_array)
# ============================================================
def load_met(filepath):
    """
    Load an ALMA .met file and return clean (speed, direction) arrays.
    Missing values (-999) are dropped.
    """
    rows = []
    with open(filepath, 'r') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) <= max(COL_SPEED, COL_DIRECTION):
                continue  # skip malformed rows
            try:
                spd = float(parts[COL_SPEED])
                drn = float(parts[COL_DIRECTION])
            except ValueError:
                continue
            rows.append((spd, drn))

    arr = np.array(rows, dtype=float)
    if arr.size == 0:
        return np.array([]), np.array([])

    speed     = arr[:, 0]
    direction = arr[:, 1]

    # Remove missing values
    valid = (speed != MISSING) & (direction != MISSING) & (speed >= 0)
    return speed[valid], direction[valid]

print("Helper loaded ✔")

In [ ]:
# ============================================================
# HELPER: draw one wind rose scatter plot
# ============================================================
def plot_wind_rose(speed, direction, title, vmin=None, vmax=None, figsize=(6, 6)):
    """
    Draw a polar scatter wind rose.

    Parameters
    ----------
    speed     : array of wind speeds (m/s)  → radial distance
    direction : array of met directions (°) → angle (N=0, clockwise)
    title     : plot title string
    vmin/vmax : colorbar range (None = auto from data)
    """
    if len(speed) == 0:
        print(f"  [SKIP] No valid data for: {title}")
        return

    # Convert meteorological direction → math angle for polar axes
    # Meteorological: 0° = North, clockwise
    # ax.set_theta_zero_location('N') + set_theta_direction(-1) handles this,
    # so we just pass direction in radians directly.
    theta = np.radians(direction)

    _vmin = vmin if vmin is not None else 0
    _vmax = vmax if vmax is not None else np.nanmax(speed)

    fig = plt.figure(figsize=figsize)
    ax  = fig.add_subplot(111, polar=True)

    sc = ax.scatter(
        theta,
        speed,
        c=speed,
        cmap=CMAP,
        vmin=_vmin,
        vmax=_vmax,
        s=DOT_SIZE,
        alpha=ALPHA,
        linewidths=0
    )

    # North at top, clockwise (meteorological convention)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)

    # Cardinal direction labels
    ax.set_thetagrids([0, 45, 90, 135, 180, 225, 270, 315],
                      ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'])

    # Radial axis label
    ax.set_ylabel("", labelpad=30)
    ax.set_rlabel_position(45)

    # Title & stats
    n_pts   = len(speed)
    mean_sp = np.mean(speed)
    max_sp  = np.max(speed)
    ax.set_title(
        f"{title}\nn={n_pts:,}  |  mean={mean_sp:.2f} m/s  |  max={max_sp:.2f} m/s",
        va='bottom', pad=20, fontsize=11, fontweight='bold'
    )

    # Colorbar
    cbar = plt.colorbar(sc, ax=ax, pad=0.12, shrink=0.7)
    cbar.set_label("Wind Speed (m/s)", fontsize=9)

    plt.tight_layout()
    plt.show()

print("Plot helper loaded ✔")

---
## 📅 Monthly Wind Roses (24 plots — one per .met file)

Files are named `wsb_YYММ.met` where `YY` is the year index (01=2001, 02=2002) and `MM` is the month (01–12).

In [ ]:
# ============================================================
# LOAD ALL FILES AND COLLECT DATA
# ============================================================
all_speed     = []
all_direction = []
monthly_data  = []   # list of dicts: {label, speed, direction}

for filepath in files:
    basename = os.path.basename(filepath)          # e.g. wsb_0101.met
    code     = basename.replace('wsb_', '').replace('.met', '')  # e.g. '0101'
    yr_code  = code[:2]   # '01' or '02'
    mo_code  = code[2:]   # '01' .. '12'

    year  = YEAR_MAP.get(yr_code, f'20{yr_code}')
    month = MONTH_NAMES.get(mo_code, mo_code)
    label = f"ALMA Ground Station — {month} {year}"

    spd, drn = load_met(filepath)

    monthly_data.append({
        'label':     label,
        'speed':     spd,
        'direction': drn,
        'year':      year,
        'month':     month
    })

    if len(spd) > 0:
        all_speed.append(spd)
        all_direction.append(drn)

    print(f"  {basename}  →  {label}  |  {len(spd)} valid points")

# Compute global vmax for consistent colorbar across all plots
global_vmax = np.nanmax(np.concatenate(all_speed)) if all_speed else 20
print(f"\nGlobal max speed: {global_vmax:.2f} m/s  (used for colorbar scale)")

In [ ]:
# ============================================================
# PLOT 1–24: ONE WIND ROSE PER MONTH
# ============================================================
for entry in monthly_data:
    plot_wind_rose(
        speed     = entry['speed'],
        direction = entry['direction'],
        title     = entry['label'],
        vmin      = 0,
        vmax      = global_vmax
    )

---
## 🌐 Combined Wind Rose (Plot 25 — All Data 2001–2002)

In [ ]:
# ============================================================
# PLOT 25: COMBINED — ALL DATA 2001 + 2002
# ============================================================
combined_speed     = np.concatenate(all_speed)
combined_direction = np.concatenate(all_direction)

print(f"Combined dataset: {len(combined_speed):,} valid data points")
print(f"  Speed  — mean: {np.mean(combined_speed):.2f} m/s  |  max: {np.max(combined_speed):.2f} m/s")
print(f"  Direction — mean: {np.mean(combined_direction):.1f}°")

plot_wind_rose(
    speed     = combined_speed,
    direction = combined_direction,
    title     = "ALMA Ground Station — Combined (2001 + 2002, All Months)",
    vmin      = 0,
    vmax      = global_vmax,
    figsize   = (7, 7)   # slightly larger for the summary plot
)

---
## 📊 Summary Statistics Table

In [ ]:
# ============================================================
# SUMMARY TABLE
# ============================================================
rows = []
for entry in monthly_data:
    spd = entry['speed']
    if len(spd) == 0:
        rows.append({
            'Period':   f"{entry['month']} {entry['year']}",
            'N':        0,
            'Mean (m/s)': np.nan,
            'Std (m/s)':  np.nan,
            'Max (m/s)':  np.nan,
            'P50 (m/s)':  np.nan,
            'P90 (m/s)':  np.nan,
            'P99 (m/s)':  np.nan,
        })
    else:
        rows.append({
            'Period':     f"{entry['month']} {entry['year']}",
            'N':          len(spd),
            'Mean (m/s)': round(np.mean(spd), 3),
            'Std (m/s)':  round(np.std(spd), 3),
            'Max (m/s)':  round(np.max(spd), 3),
            'P50 (m/s)':  round(np.percentile(spd, 50), 3),
            'P90 (m/s)':  round(np.percentile(spd, 90), 3),
            'P99 (m/s)':  round(np.percentile(spd, 99), 3),
        })

# Add combined row
rows.append({
    'Period':     'ALL 2001–2002',
    'N':          len(combined_speed),
    'Mean (m/s)': round(np.mean(combined_speed), 3),
    'Std (m/s)':  round(np.std(combined_speed), 3),
    'Max (m/s)':  round(np.max(combined_speed), 3),
    'P50 (m/s)':  round(np.percentile(combined_speed, 50), 3),
    'P90 (m/s)':  round(np.percentile(combined_speed, 90), 3),
    'P99 (m/s)':  round(np.percentile(combined_speed, 99), 3),
})

summary_df = pd.DataFrame(rows)
summary_df